<a href="https://colab.research.google.com/github/KumarM679/Flyrank-intership/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duaaanadeem/ML-workspace/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [7]:
# Rule:
# Score pages based on:
# 1. Low CTR (higher priority)
# 2. Old content (staleness)
# 3. High traffic volume (opportunity)

def calculate_score(ctr, age_days, impressions):
    score = 0
    reasons = []

    # CTR signal
    if ctr < 2:
        score += 40
        reasons.append("LOW_CTR")

    # Staleness signal
    if age_days > 180:
        score += 30
        reasons.append("OLD_CONTENT")

    # Volume signal
    if impressions > 10000:
        score += 30
        reasons.append("HIGH_VOLUME")

    # If no strong signal found
    if len(reasons) == 0:
        reasons.append("MIXED_SIGNAL")

    # Use only one main reason code
    reason_code = reasons[0]

    # Action label
    if score >= 70:
        action = "QUICK_WIN"
    elif score >= 40:
        action = "INVESTIGATE"
    else:
        action = "IGNORE"

    return score, reason_code, action


# Example test
page_score = calculate_score(
    ctr=1.5,
    age_days=250,
    impressions=15000
)

print("Score:", page_score[0])
print("Reason:", page_score[1])
print("Action:", page_score[2])

Score: 100
Reason: LOW_CTR
Action: QUICK_WIN


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
import pandas as pd
import os


# Sample input data
data = {
    "page": [
        "page_A",
        "page_B",
        "page_C",
        "page_D",
        "page_E",
        "page_F"
    ],
    "ctr": [
        1.2,
        3.5,
        1.8,
        5.0,
        0.9,
        2.5
    ],
    "age_days": [
        250,
        60,
        200,
        30,
        300,
        120
    ],
    "impressions": [
        15000,
        5000,
        20000,
        3000,
        25000,
        12000
    ]
}


df = pd.DataFrame(data)


# Rule function
def apply_rule(row):

    score = 0
    reason = "MIXED_SIGNAL"

    # Low CTR
    if row["ctr"] < 2:
        score += 40
        reason = "LOW_CTR"

    # Old content
    if row["age_days"] > 180:
        score += 30
        if reason == "MIXED_SIGNAL":
            reason = "OLD_CONTENT"

    # High volume
    if row["impressions"] > 10000:
        score += 30
        if reason == "MIXED_SIGNAL":
            reason = "HIGH_VOLUME"


    # Action label
    if score >= 70:
        action = "QUICK_WIN"
    elif score >= 40:
        action = "INVESTIGATE"
    else:
        action = "IGNORE"


    return pd.Series([score, reason, action])


# Apply scoring rule
df[["score", "reason_code", "action"]] = df.apply(
    apply_rule,
    axis=1
)


# Rank by highest score
df = df.sort_values(
    by="score",
    ascending=False
)

# Add rank column
df.insert(
    0,
    "rank",
    range(1, len(df)+1)
)


# Create output folder
os.makedirs(
    "work/outputs",
    exist_ok=True
)


# Write CSV file
output_path = "work/outputs/baseline_action_score.csv"

df.to_csv(
    output_path,
    index=False
)


print("CSV created successfully!")
print("Saved at:", output_path)


# Display ranked queue
df

CSV created successfully!
Saved at: work/outputs/baseline_action_score.csv


,rank,page,ctr,age_days,impressions,score,reason_code,action
0,1,page_A,1.2,250,15000,100,LOW_CTR,QUICK_WIN
2,2,page_C,1.8,200,20000,100,LOW_CTR,QUICK_WIN
4,3,page_E,0.9,300,25000,100,LOW_CTR,QUICK_WIN
5,4,page_F,2.5,120,12000,30,HIGH_VOLUME,IGNORE
1,5,page_B,3.5,60,5000,0,MIXED_SIGNAL,IGNORE
3,6,page_D,5.0,30,3000,0,MIXED_SIGNAL,IGNORE


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
import pandas as pd


# Load ranked queue
df = pd.read_csv(
    "work/outputs/baseline_action_score.csv"
)


# Select top 20 items
top20 = df.head(20).copy()


# Add confidence notes
def confidence_note(row):

    if row["action"] == "QUICK_WIN":
        return "High confidence because multiple signals show improvement opportunity"

    elif row["action"] == "INVESTIGATE":
        return "Medium confidence because signals need manual checking"

    else:
        return "Low confidence because no strong problem signal found"



# Add what could make it wrong
def wrong_reason(row):

    if row["reason_code"] == "LOW_CTR":
        return "CTR may be low because the keyword is not relevant"

    elif row["reason_code"] == "OLD_CONTENT":
        return "Old content may still be useful and accurate"

    elif row["reason_code"] == "HIGH_VOLUME":
        return "High traffic does not always mean improvement is needed"

    else:
        return "Data may not show the complete picture"



# Create review columns
top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    wrong_reason,
    axis=1
)


# Select final review columns
review = top20[
    [
        "rank",
        "page",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]


# Display Top-20 review
print(review.to_markdown(index=False))

|   rank | page   | action    | reason_code   | confidence_note                                                       | what_would_make_it_wrong                                |
|-------:|:-------|:----------|:--------------|:----------------------------------------------------------------------|:--------------------------------------------------------|
|      1 | page_A | QUICK_WIN | LOW_CTR       | High confidence because multiple signals show improvement opportunity | CTR may be low because the keyword is not relevant      |
|      2 | page_C | QUICK_WIN | LOW_CTR       | High confidence because multiple signals show improvement opportunity | CTR may be low because the keyword is not relevant      |
|      3 | page_E | QUICK_WIN | LOW_CTR       | High confidence because multiple signals show improvement opportunity | CTR may be low because the keyword is not relevant      |
|      4 | page_F | IGNORE    | HIGH_VOLUME   | Low confidence because no strong problem signal found         

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
allowed_signals = [
    "ctr",
    "age_days",
    "impressions"
]

print("Signals used:", allowed_signals)

print("Leakage Check:")
print("✓ No product flags included")
print("✓ No future window data included")
print("✓ Only current signals used")

Signals used: ['ctr', 'age_days', 'impressions']
Leakage Check:
✓ No product flags included
✓ No future window data included
✓ Only current signals used


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.